In [1]:
from pathlib import Path
import platform
import re

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / "Data/Trafficdata/traffic_23.csv").exists() and (project_root.parent / "Data/Trafficdata/traffic_23.csv").exists():
    project_root = project_root.parent

if platform.system() == "Darwin":
    pd.set_option("display.unicode.east_asian_width", True)
    pd.set_option("display.unicode.ambiguous_as_wide", True)


In [2]:
hour_cols = [f"{hour}시" for hour in range(6, 24)]
date_type_order = ["평일", "주말"]

traffic_paths = [
    project_root / "Data/Trafficdata/traffic_23.csv",
    project_root / "Data/Trafficdata/traffic_24.csv",
    project_root / "Data/Trafficdata/traffic_25.csv",
]
selected_bridges = ["서강대교", "마포대교", "원효대교"]

traffic_frames = []
for path in traffic_paths:
    df = pd.read_csv(path)
    df["일자"] = pd.to_datetime(df["일자"].astype(str), format="%Y%m%d")
    df["날짜타입"] = df["일자"].dt.weekday.map(lambda x: "주말" if x >= 5 else "평일")
    traffic_frames.append(df[df["지점명"].isin(selected_bridges)][["지점명", "날짜타입", "방향"] + hour_cols].copy())

traffic_df = pd.concat(traffic_frames, ignore_index=True)
traffic_station_direction_avg = traffic_df.groupby(["지점명", "날짜타입", "방향"], as_index=False)[hour_cols].mean()
traffic_station_avg = traffic_station_direction_avg.groupby(["지점명", "날짜타입"], as_index=False)[hour_cols].mean()
traffic_profile_df = (
    traffic_station_avg.groupby("날짜타입")[hour_cols]
    .median()
    .reindex(date_type_order)
)


In [3]:
metro_paths = [
    project_root / "Data/Metro/서울교통공사_역별 시간대별 승하차인원(23.1~23.12) (1).csv",
    project_root / "Data/Metro/서울교통공사_역별 시간대별 승하차인원(24.1~24.12) (1).csv",
]
selected_stations = ["여의도", "여의나루"]


def extract_start_hour(column_name):
    if "이전" in column_name or "이후" in column_name:
        return None
    numbers = re.findall(r"\d{2}", column_name)
    if len(numbers) >= 2:
        start_hour = int(numbers[0])
        if 6 <= start_hour <= 23:
            return start_hour
    return None


metro_frames = []
for path in metro_paths:
    df = pd.read_csv(path, encoding="cp949")
    station_col = "역명"
    direction_col = "승하차구분" if "승하차구분" in df.columns else "구분"
    date_col = "수송일자" if "수송일자" in df.columns else "날짜"

    hour_map = {}
    for column in df.columns:
        start_hour = extract_start_hour(column)
        if start_hour is not None:
            hour_map[column] = f"{start_hour}시"

    keep_cols = [station_col, date_col, direction_col] + list(hour_map.keys())
    metro_df = df[df[station_col].isin(selected_stations)][keep_cols].copy()
    metro_df[date_col] = pd.to_datetime(metro_df[date_col])
    metro_df["날짜타입"] = metro_df[date_col].dt.weekday.map(lambda x: "주말" if x >= 5 else "평일")
    metro_df = metro_df.rename(columns=hour_map)[[station_col, "날짜타입", direction_col] + hour_cols]
    metro_frames.append(metro_df)

metro_df = pd.concat(metro_frames, ignore_index=True)
metro_station_direction_avg = metro_df.groupby([station_col, "날짜타입", direction_col], as_index=False)[hour_cols].mean()
metro_station_avg = metro_station_direction_avg.groupby([station_col, "날짜타입"], as_index=False)[hour_cols].mean()
metro_profile_df = (
    metro_station_avg.groupby("날짜타입")[hour_cols]
    .median()
    .reindex(date_type_order)
)


In [4]:
combined_raw_df = (traffic_profile_df + metro_profile_df) / 2


def minmax_scale_row(row):
    row_min = row.min()
    row_max = row.max()
    if pd.isna(row_min) or pd.isna(row_max):
        return pd.Series(pd.NA, index=row.index, dtype="float64")
    if row_max == row_min:
        return pd.Series(1.0, index=row.index, dtype="float64")
    return (row - row_min) / (row_max - row_min)


scaled_df = combined_raw_df.apply(minmax_scale_row, axis=1)
weight_df = scaled_df.div(scaled_df.sum(axis=1), axis=0)
weight_df.index.name = "날짜타입"
weight_df = weight_df.reindex(date_type_order)

display(weight_df)


,6시,7시,8시,9시,10시,11시,12시,13시,14시,15시,16시,17시,18시,19시,20시,21시,22시,23시
날짜타입,,,,,,,,,,,,,,,,,,
평일,0.01547,0.094995,0.138634,0.064270,0.04028,0.040673,0.03588,0.044357,0.044833,0.050759,0.061228,0.108140,0.123022,0.053690,0.038653,0.029200,0.015916,0.000000
주말,0.00000,0.008019,0.024231,0.037293,0.05675,0.066671,0.07341,0.083311,0.092339,0.097481,0.101299,0.095941,0.077005,0.060306,0.057394,0.042853,0.024164,0.001532
